In [1]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(
    file_name,
    compression="gzip",
    low_memory=False
)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Saving medimate_modeling_base.csv.gz to medimate_modeling_base.csv (1).gz
Dataset shape: (494862, 25)
Columns: ['PT_CODE', 'WL_ID_CODE', 'INIT_DATE', 'listing_year', 'LISTING_CTR_CODE', 'INIT_AGE', 'GENDER', 'ETHCAT', 'ABO', 'INIT_CPRA', 'INIT_CPRA_missing', 'ON_DIALYSIS', 'BMI_TCR_clean', 'BMI_missing', 'FUNC_STAT_TCR_clean', 'FUNC_STAT_missing', 'INIT_STAT', 'REGION', 'A2A2B_status', 'outcome', 'event_adverse', 'event_transplant', 'censored', 'survival_valid', 'survival_time_days_clean']


In [2]:
HORIZON = 365

valid = df["survival_valid"] == 1
time = df["survival_time_days_clean"]

positive = (
    valid &
    (df["event_adverse"] == 1) &
    (time <= HORIZON)
)

known_negative = (
    valid &
    (df["event_adverse"] == 0) &
    (df["censored"] == 0) &
    (df["outcome"] != "unknown")
)

late_adverse = (
    valid &
    (df["event_adverse"] == 1) &
    (time > HORIZON)
)

waiting_long_enough = (
    valid &
    (df["censored"] == 1) &
    (time >= HORIZON)
)

df["adverse_1yr"] = np.nan
df.loc[positive, "adverse_1yr"] = 1
df.loc[
    known_negative | late_adverse | waiting_long_enough,
    "adverse_1yr"
] = 0

cohort = df[df["adverse_1yr"].notna()].copy()

negatives = cohort[cohort["adverse_1yr"] == 0]
early_negatives = negatives[
    negatives["survival_time_days_clean"] < HORIZON
]

print("Total cohort:", len(cohort))
print("Positive labels:", int(cohort["adverse_1yr"].sum()))
print("Negative labels:", len(negatives))

print("\nALL NEGATIVES BY OUTCOME")
print(negatives["outcome"].value_counts())

print("\nNEGATIVES WITH LESS THAN 365 DAYS FOLLOW-UP")
print(early_negatives["outcome"].value_counts())

print("\nTotal early negatives:", len(early_negatives))


Total cohort: 450764
Positive labels: 17587
Negative labels: 433177

ALL NEGATIVES BY OUTCOME
outcome
transplanted              234429
still_waiting              64076
removed_administrative     45331
transplanted_elsewhere     37243
removed_too_sick           27666
died                       24432
Name: count, dtype: int64

NEGATIVES WITH LESS THAN 365 DAYS FOLLOW-UP
outcome
transplanted              139721
transplanted_elsewhere     14883
removed_administrative     10877
Name: count, dtype: int64

Total early negatives: 165481


In [3]:

# Compare classification cohort strategies

HORIZON = 365

# Strategy A
cohort_A = cohort.copy()

# Strategy B: Require full follow-up for negatives
# Keep all observed adverse events within 365 days
# Keep negatives only if observed for >= 365 days
cohort_B = cohort[
    (cohort["adverse_1yr"] == 1) |
    (cohort["survival_time_days_clean"] >= HORIZON)
].copy()

# Strategy C: Keep early transplants as competing events,
# but exclude early administrative removals
early_admin = (
    (cohort["outcome"] == "removed_administrative") &
    (cohort["survival_time_days_clean"] < HORIZON)
)

cohort_C = cohort[~early_admin].copy()

# Compare sizes and class balance
results = []

for name, data in [
    ("A: Baseline", cohort_A),
    ("B: Full follow-up", cohort_B),
    ("C: Competing-risk", cohort_C)
]:
    positives = int(data["adverse_1yr"].sum())
    total = len(data)

    results.append({
        "Strategy": name,
        "Records": total,
        "Excluded vs A": len(cohort_A) - total,
        "Positive cases": positives,
        "Positive rate (%)": round(
            positives / total * 100, 2
        )
    })

comparison = pd.DataFrame(results)
display(comparison)


,Strategy,Records,Excluded vs A,Positive cases,Positive rate (%)
0,A: Baseline,450764,0,17587,3.90
1,B: Full follow-up,285283,165481,17587,6.16
2,C: Competing-risk,439887,10877,17587,4.00


In [4]:
master = pd.read_csv(
    "/content/medimate_cleaned_master.csv.gz",
    compression="gzip",
    low_memory=False
)

early_admin_master = master[
    (master["outcome"] == "removed_administrative") &
    (master["survival_time_days_clean"] < 365)
].copy()

print("Early administrative removals:", len(early_admin_master))

print("\nRemoval reason codes:")
print(early_admin_master["REM_CD"].value_counts(dropna=False))

Early administrative removals: 10877

Removal reason codes:
REM_CD
9.0     6616
7.0     1891
17.0    1435
24.0     682
16.0     131
40.0     122
Name: count, dtype: int64


## Classification Cohort Strategy — Findings & Recommendation

### Objective
Determine which waitlist records should be included in the 1-year adverse-outcome classifier, particularly how to handle patients with incomplete follow-up or competing outcomes.

### Baseline Cohort
Reproduced the existing classification cohort:
- **450,764** eligible records
- **17,587** positive cases (3.90%)
- **433,177** negative cases

Patients still waiting with less than 365 days of follow-up are excluded.

### Alternative Cohort Strategies

| Strategy | Records | Positive Rate |
|---|---:|---:|
| A: Existing baseline | 450,764 | 3.90% |
| B: Require full follow-up for negatives | 285,283 | 6.16% |
| C: Exclude early administrative removals | 439,887 | 4.00% |

Requiring full follow-up removes 165,481 records, including 154,604 early transplants or transplants elsewhere. This may introduce selection bias by disproportionately excluding patients who received transplants.

### Administrative Removal Investigation
Identified 10,877 patients administratively removed before 365 days, primarily associated with removal codes 9, 7, and 17.

### Preliminary Recommendation
Retain the existing 365-day classification cohort as the baseline while clarifying the prediction target.

- Include adverse outcomes occurring within 365 days as positives.
- Include patients with at least 365 days of event-free observation as negatives.
- Exclude patients still waiting with insufficient follow-up.
- Treat early transplants as competing outcomes rather than automatically excluding them.
- Review administrative removal reasons before deciding whether to retain them as negatives.